# 01. Tối ưu không ràng buộc — Genetic Algorithm tìm Global Optimum

Genetic Algorithm tìm cực tiểu toàn cục (global optimum) của các hàm
benchmark kinh điển cho tối ưu không ràng buộc, đối chiếu với
**PyGAD** — thư viện GA có sẵn — cả về kết quả lẫn thời gian chạy.

- **Easom** — gần như bằng phẳng ($\approx 0$) ở mọi nơi trừ một hố hẹp
  duy nhất tại cực tiểu toàn cục $f(\pi,\pi)=-1$ — kiểm tra khả năng
  tìm ĐÚNG cực trị toàn cục thay vì dừng ở vùng phẳng xung quanh.
- **Rastrigin** — nhiều cực tiểu cục bộ dày đặc quanh một cực tiểu toàn cục.
- **Rosenbrock** — "thung lũng cong" hẹp, khó dò theo hơn là khó tìm.
- **Ackley** — hố lớn có gợn sóng cực tiểu cục bộ nhỏ phủ lên trên.
- **Schwefel** — hàm "lừa": vùng quanh gốc tọa độ trông tốt nhưng cực
  tiểu toàn cục thực ra nằm gần biên miền tìm kiếm.


In [1]:
import time

import numpy as np
import pygad
import sympy


# ============================================================
# GENETIC ALGORITHM CHO TỐI ƯU KHÔNG RÀNG BUỘC
# ============================================================
#
# Khác với GA có ràng buộc (xem GA_test.ipynb): ở đây fitness = f(x)
# trực tiếp, không cần quy tắc khả thi Deb hay ngưỡng epsilon giảm dần.
#
# Miền tìm kiếm [lower, upper] đóng vai trò MIỀN XÁC ĐỊNH của bài toán
# tối ưu toàn cục, nên cá thể được CẮT (clip) về đúng miền này sau mỗi
# phép lai ghép / đột biến — khác GA có ràng buộc, nơi hộp chỉ dùng để
# khởi tạo quần thể chứ không giới hạn cá thể.


# Tham số GA dùng CHUNG cho cả bản tự cài lẫn PyGAD bên dưới, để việc
# so sánh không lệch nhau vì hai bộ tham số khác nhau.
CROSSOVER_RATE = 0.9
MUTATION_RATE = 0.15
MUTATION_SCALE = 0.08
ELITE_SIZE = 2
TOURNAMENT_SIZE = 3


def genetic_algorithm_unconstrained(
    objective,
    bounds,

    population_size=100,
    generations=500,

    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    mutation_scale=MUTATION_SCALE,

    elite_size=ELITE_SIZE,
    tournament_size=TOURNAMENT_SIZE,

    seed=42,
):
    """GA mã hóa số thực cho bài toán tối ưu KHÔNG ràng buộc, tìm cực tiểu
    toàn cục của f(x) trong hộp [lower, upper]."""

    rng = np.random.default_rng(seed)

    bounds = np.asarray(bounds, dtype=float)

    lower = bounds[:, 0]
    upper = bounds[:, 1]

    variable_range = upper - lower
    n_variables = len(bounds)

    def clip(pop):
        return np.clip(pop, lower, upper)

    # --------------------------------------------------------
    # Initial population
    # --------------------------------------------------------

    population = clip(
        rng.uniform(lower, upper, size=(population_size, n_variables))
    )

    def evaluate(pop):
        return np.array([objective(individual) for individual in pop])

    # --------------------------------------------------------
    # Tournament selection (theo thứ hạng fitness)
    # --------------------------------------------------------

    def tournament_selection(rank):

        indices = rng.integers(0, population_size, size=tournament_size)

        best_index = indices[np.argmin(rank[indices])]

        return population[best_index].copy()

    # --------------------------------------------------------
    # Blend crossover
    # --------------------------------------------------------

    def crossover(parent1, parent2):

        if rng.random() > crossover_rate:
            return parent1.copy(), parent2.copy()

        alpha = rng.uniform(-0.25, 1.25, size=n_variables)

        child1 = alpha * parent1 + (1 - alpha) * parent2
        child2 = alpha * parent2 + (1 - alpha) * parent1

        return child1, child2

    # --------------------------------------------------------
    # Gaussian mutation
    # --------------------------------------------------------

    def mutate(child):

        mutation_mask = rng.random(n_variables) < mutation_rate

        if np.any(mutation_mask):
            child[mutation_mask] += rng.normal(
                loc=0,
                scale=mutation_scale * variable_range[mutation_mask],
            )

        return child

    # --------------------------------------------------------
    # Evolution
    # --------------------------------------------------------

    history = []

    start_time = time.perf_counter()

    fitness = evaluate(population)

    best_index = int(np.argmin(fitness))
    best_solution = population[best_index].copy()
    best_fitness = fitness[best_index]

    for generation in range(generations):

        order = np.argsort(fitness)

        rank = np.empty(population_size, dtype=np.int64)
        rank[order] = np.arange(population_size)

        current_best = int(np.argmin(fitness))
        if fitness[current_best] < best_fitness:
            best_fitness = fitness[current_best]
            best_solution = population[current_best].copy()

        history.append(best_fitness)

        # Elitism
        new_population = [
            population[i].copy()
            for i in order[:elite_size]
        ]

        # Sinh thế hệ tiếp theo
        while len(new_population) < population_size:

            parent1 = tournament_selection(rank)
            parent2 = tournament_selection(rank)

            child1, child2 = crossover(parent1, parent2)

            new_population.append(mutate(child1))

            if len(new_population) < population_size:
                new_population.append(mutate(child2))

        population = clip(np.asarray(new_population))
        fitness = evaluate(population)

    current_best = int(np.argmin(fitness))
    if fitness[current_best] < best_fitness:
        best_fitness = fitness[current_best]
        best_solution = population[current_best].copy()

    elapsed_time = time.perf_counter() - start_time

    # Thế hệ sớm nhất mà nghiệm tốt nhất (best_fitness cuối cùng) đã đạt
    # được — chỉ là một chỉ số báo cáo, KHÔNG dừng vòng lặp sớm.
    generations_run = int(np.argmin(history)) + 1

    return {
        "x": best_solution,
        "fun": best_fitness,
        "time": elapsed_time,
        "history": history,
        "generations": generations,
        "generations_run": generations_run,
        "seed": seed,
    }


# ============================================================
# PYGAD — THƯ VIỆN GA CÓ SẴN, DÙNG LÀM MỐC SO SÁNH
# ============================================================
#
# Dùng LẠI đúng công thức lai ghép (blend crossover) và đột biến
# (Gaussian) của GA tự cài ở trên, qua hàm tùy chỉnh — PyGAD không có
# sẵn hai toán tử này (chỉ có single-point/two-points/uniform/scattered/
# sbx cho crossover, và random/swap/inversion/scramble cho mutation).
# Nhờ vậy khác biệt còn lại chỉ nằm ở KIẾN TRÚC vòng lặp của thư viện —
# PyGAD chọn sẵn một pool `num_parents_mating` cha mẹ mỗi thế hệ (qua
# tournament) rồi lai ghép tuần tự trong pool đó, còn GA tự cài chọn 2
# cha mẹ MỚI qua tournament cho MỖI phép lai — không phải khác biệt về
# công thức toán học của từng toán tử.
#
# PyGAD tối đa hóa fitness, nên fitness = -f(x) để tương đương tối
# thiểu hóa f(x). gene_space={'low','high'} vừa khởi tạo vừa giữ mỗi
# gene trong đúng miền tìm kiếm, giống clip() ở GA tự cài.

def run_pygad(objective, bounds, n_variables, population_size, generations, seed):
    """Chạy PyGAD trên cùng bài toán, trả về kết quả cùng định dạng với
    genetic_algorithm_unconstrained để show_comparison dùng chung.

    Cả lai ghép (blend crossover) lẫn đột biến (Gaussian) đều dùng lại
    ĐÚNG công thức của GA tự cài, qua hàm tùy chỉnh truyền vào
    crossover_type / mutation_type — không dùng toán tử mặc định của
    PyGAD, để hai bản cài đặt xử lý cá thể theo cùng một cách."""

    bounds_arr = np.asarray(bounds, dtype=float)
    lower = bounds_arr[:, 0]
    upper = bounds_arr[:, 1]
    variable_range = upper - lower

    rng = np.random.default_rng(seed)

    def fitness_func(ga_instance, solution, solution_idx):
        return -objective(solution)

    def gaussian_mutation(offspring, ga_instance):
        for i in range(offspring.shape[0]):
            mask = rng.random(n_variables) < MUTATION_RATE
            if np.any(mask):
                offspring[i, mask] += rng.normal(
                    0, MUTATION_SCALE * variable_range[mask]
                )
            offspring[i] = np.clip(offspring[i], lower, upper)
        return offspring

    def blend_crossover(parents, offspring_size, ga_instance):
        num_offspring, num_genes = offspring_size
        n_parents = parents.shape[0]
        offspring = np.empty(offspring_size, dtype=float)

        for k in range(num_offspring):
            parent1 = parents[k % n_parents]
            parent2 = parents[(k + 1) % n_parents]

            if rng.random() > CROSSOVER_RATE:
                offspring[k] = parent1.copy()
                continue

            alpha = rng.uniform(-0.25, 1.25, size=num_genes)
            offspring[k] = alpha * parent1 + (1 - alpha) * parent2

        return np.clip(offspring, lower, upper)

    start_time = time.perf_counter()

    ga_instance = pygad.GA(
        num_generations=generations,
        num_parents_mating=max(2, population_size // 2),
        fitness_func=fitness_func,
        sol_per_pop=population_size,
        num_genes=n_variables,
        gene_space=[{"low": b[0], "high": b[1]} for b in bounds],
        parent_selection_type="tournament",
        K_tournament=TOURNAMENT_SIZE,
        crossover_type=blend_crossover,
        mutation_type=gaussian_mutation,
        keep_elitism=ELITE_SIZE,
        random_seed=seed,
        suppress_warnings=True,
    )
    ga_instance.run()

    solution, _fitness, _index = ga_instance.best_solution()
    solution = np.asarray(solution, dtype=float)

    elapsed_time = time.perf_counter() - start_time

    # PyGAD tự tính sẵn thế hệ (0-indexed) mà nghiệm tốt nhất đạt được —
    # cùng ý nghĩa với generations_run của GA tự cài, +1 cho khớp quy ước.
    generations_run = int(ga_instance.best_solution_generation) + 1

    return {
        "x": solution,
        "fun": objective(solution),
        "time": elapsed_time,
        "generations_run": generations_run,
    }


print(f"numpy {np.__version__} | pygad {pygad.__version__} | sympy {sympy.__version__}")


# ------------------------------------------------------------------
# Hiển thị dạng ký hiệu toán học (LaTeX)
# ------------------------------------------------------------------
from IPython.display import Markdown, display


def _num(value, digits=10):
    """Số dạng LaTeX; chuyển sang ký hiệu khoa học khi quá lớn hoặc quá nhỏ."""
    if not np.isfinite(value):
        return r"\infty" if value > 0 else r"-\infty"
    if value != 0 and (abs(value) >= 1e6 or abs(value) < 1e-4):
        mantissa, exponent = f"{value:.4e}".split("e")
        return mantissa + r" \times 10^{" + str(int(exponent)) + "}"
    return f"{value:.{digits}f}"


def _num_bound(value):
    """Số dạng LaTeX cho cận tìm kiếm — bỏ số 0 thừa: -10 thay vì -10.0000."""
    text = _num(value, 4)
    if "." in text and "times" not in text:
        text = text.rstrip("0").rstrip(".")
    return text


def show_problem(objective_expr, variables, bounds):
    """Phát biểu bài toán tối ưu không ràng buộc và miền tìm kiếm."""
    bien = ", ".join(sympy.latex(v) for v in variables)
    mien = ", \\ ".join(
        f"{_num_bound(b[0])} \\le {sympy.latex(v)} \\le {_num_bound(b[1])}"
        for v, b in zip(variables, bounds)
    )
    display(Markdown(
        "$$\n\\underset{" + bien + r"}{\text{minimize}} \quad f\left("
        + bien + r"\right) = " + sympy.latex(objective_expr) + "\n$$\n\n"
        + "Miền tìm kiếm: $" + mien + "$"
    ))


def show_comparison(results, variables):
    """Bảng so sánh nhiều phương pháp: f*, thời gian, số thế hệ thỏa mãn,
    tọa độ nghiệm.

    `results` là danh sách [(tên, result), ...]. Result không có khóa
    "generations_run" thì để trống ô đó."""
    cot_bien = " | ".join("$" + sympy.latex(v) + "$" for v in variables)

    dong = [
        r"| Phương pháp | $f^{*}$ | Thời gian (s) | Số thế hệ thỏa mãn | "
        + cot_bien + " |",
        "|---|---|---|---|" + "---|" * len(variables),
    ]

    for ten, r in results:
        toado = " | ".join("$" + _num(x) + "$" for x in r["x"])
        the_he = str(r["generations_run"]) if "generations_run" in r else ""
        dong.append("| " + ten + " | $" + _num(r["fun"]) + "$ | $"
                    + _num(r["time"], 6) + "$ | " + the_he + " | " + toado + " |")

    display(Markdown("\n".join(dong)))


numpy 1.26.4 | pygad 3.7.0 | sympy 1.14.0


In [2]:
x, y = sympy.symbols("x y")
variables_xy = [x, y]

# Tham số GA dùng cho toàn bộ benchmark bên dưới.
POPULATION_SIZE = 500
GENERATIONS = 50
N_RUNS = 3

# Năm hàm benchmark kinh điển, đặt thủ công (không qua parser) —
# miền tìm kiếm lấy theo giá trị chuẩn thường dùng trong tài liệu.
BENCHMARKS = {
    "Easom": {
        "expr": (
            -sympy.cos(x) * sympy.cos(y)
            * sympy.exp(-((x - sympy.pi) ** 2 + (y - sympy.pi) ** 2))
        ),
        "bounds": [(-100.0, 100.0), (-100.0, 100.0)],
        "optimum": r"f(\pi, \pi) = -1",
    },
    "Rastrigin": {
        "expr": (
            20 + x**2 - 10 * sympy.cos(2 * sympy.pi * x)
            + y**2 - 10 * sympy.cos(2 * sympy.pi * y)
        ),
        "bounds": [(-5.12, 5.12), (-5.12, 5.12)],
        "optimum": "f(0, 0) = 0",
    },
    "Rosenbrock": {
        "expr": (1 - x) ** 2 + 100 * (y - x**2) ** 2,
        "bounds": [(-5.0, 10.0), (-5.0, 10.0)],
        "optimum": "f(1, 1) = 0",
    },
    "Ackley": {
        "expr": (
            -20 * sympy.exp(-0.2 * sympy.sqrt(0.5 * (x**2 + y**2)))
            - sympy.exp(0.5 * (sympy.cos(2 * sympy.pi * x) + sympy.cos(2 * sympy.pi * y)))
            + 20 + sympy.E
        ),
        "bounds": [(-32.768, 32.768), (-32.768, 32.768)],
        "optimum": "f(0, 0) = 0",
    },
    "Schwefel": {
        "expr": (
            418.9829 * 2
            - x * sympy.sin(sympy.sqrt(sympy.Abs(x)))
            - y * sympy.sin(sympy.sqrt(sympy.Abs(y)))
        ),
        "bounds": [(-500.0, 500.0), (-500.0, 500.0)],
        "optimum": r"f(420.9687, 420.9687) \approx 0",
    },
}

for name, spec in BENCHMARKS.items():

    expr = spec["expr"]
    bounds = spec["bounds"]

    objective_raw = sympy.lambdify(variables_xy, expr, modules="numpy")

    def objective(v):
        try:
            value = float(np.asarray(objective_raw(*v)).reshape(()))
            if np.isfinite(value):
                return value
        except Exception:
            pass
        return np.inf

    display(Markdown(f"### {name} — nghiệm đúng: ${spec['optimum']}$"))
    show_problem(expr, variables_xy, bounds)

    ga_runs = [
        genetic_algorithm_unconstrained(
            objective, bounds,
            population_size=POPULATION_SIZE,
            generations=GENERATIONS,
            seed=42 + i,
        )
        for i in range(N_RUNS)
    ]
    ga_result = min(ga_runs, key=lambda r: r["fun"])

    pygad_runs = [
        run_pygad(
            objective, bounds, len(variables_xy),
            population_size=POPULATION_SIZE,
            generations=GENERATIONS,
            seed=42 + i,
        )
        for i in range(N_RUNS)
    ]
    pygad_result = min(pygad_runs, key=lambda r: r["fun"])

    show_comparison(
        [
            ("Genetic Algorithm", ga_result),
            ("PyGAD", pygad_result),
        ],
        variables_xy,
    )


### Easom — nghiệm đúng: $f(\pi, \pi) = -1$

$$
\underset{x, y}{\text{minimize}} \quad f\left(x, y\right) = - e^{- \left(x - \pi\right)^{2} - \left(y - \pi\right)^{2}} \cos{\left(x \right)} \cos{\left(y \right)}
$$

Miền tìm kiếm: $-100 \le x \le 100, \ -100 \le y \le 100$

| Phương pháp | $f^{*}$ | Thời gian (s) | Số thế hệ thỏa mãn | $x$ | $y$ |
|---|---|---|---|---|---|
| Genetic Algorithm | $-1.0000000000$ | $3.095265$ | 50 | $3.1415926394$ | $3.1415926529$ |
| PyGAD | $-1.0000000000$ | $3.895000$ | 51 | $3.1415926575$ | $3.1415926605$ |

### Rastrigin — nghiệm đúng: $f(0, 0) = 0$

$$
\underset{x, y}{\text{minimize}} \quad f\left(x, y\right) = x^{2} + y^{2} - 10 \cos{\left(2 \pi x \right)} - 10 \cos{\left(2 \pi y \right)} + 20
$$

Miền tìm kiếm: $-5.12 \le x \le 5.12, \ -5.12 \le y \le 5.12$

...

$$
\underset{x, y}{\text{minimize}} \quad f\left(x, y\right) = - e^{0.5 \cos{\left(2 \pi x \right)} + 0.5 \cos{\left(2 \pi y \right)}} + e + 20 - 20 e^{- 0.2 \sqrt{0.5 x^{2} + 0.5 y^{2}}}
$$

Miền tìm kiếm: $-32.768 \le x \le 32.768, \ -32.768 \le y \le 32.768$

| Phương pháp | $f^{*}$ | Thời gian (s) | Số thế hệ thỏa mãn | $x$ | $y$ |
|---|---|---|---|---|---|
| Genetic Algorithm | $5.9089 \times 10^{-11}$ | $3.411575$ | 50 | $-1.6166 \times 10^{-11}$ | $-1.3232 \times 10^{-11}$ |
| PyGAD | $2.9921 \times 10^{-11}$ | $4.787682$ | 50 | $-1.0552 \times 10^{-11}$ | $7.5011 \times 10^{-13}$ |

### Schwefel — nghiệm đúng: $f(420.9687, 420.9687) \approx 0$

$$
\underset{x, y}{\text{minimize}} \quad f\left(x, y\right) = - x \sin{\left(\sqrt{\left|{x}\right|} \right)} - y \sin{\left(\sqrt{\left|{y}\right|} \right)} + 837.9658
$$

Miền tìm kiếm: $-500 \le x \le 500, \ -500 \le y \le 500$

| Phương pháp | $f^{*}$ | Thời gian (s) | Số thế hệ thỏa mãn | $x$ | $y$ |
|---|---|---|---|---|---|
| Genetic Algorithm | $2.5455 \times 10^{-5}$ | $3.056131$ | 47 | $420.9687461844$ | $420.9687460969$ |
| PyGAD | $2.5455 \times 10^{-5}$ | $3.946509$ | 40 | $420.9687463690$ | $420.9687461346$ |